# FTR Auctions

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from datetime import datetime, timedelta

import random

## data import

### auction price

In [ ]:
offset = (
    pd.Timestamp.today().normalize()
    - pd.Timestamp("2026-06-01")
).days

today = (pd.Timestamp.today() - pd.Timedelta(days=offset)).normalize()

today_str = (datetime.today() - timedelta(days=offset)).strftime("%m%d") 

FTRAuctions = pd.read_csv(f'data/FTRAuction/FTRAuctions_{today_str}.csv', skiprows=[1])

FTRAuctions.rename(columns={"EndDate": "AuctionEndDate"}, inplace=True)

FTRAuctions.head()

,AuctionKey,ISOCode,AuctionName,StartDate,AuctionEndDate,AuctionRound,AuctionDate,AuctionStatusCode,ResultsPostedDate,ISOAuctionName
0,3205,PJM,Long Term 2026-29 RD 5,2026-03-03 00:00:00.000,2026-03-05 00:00:00.000,5,2026-06-01 00:00:00.000,CLOSED,2026-03-11 00:00:00.000,26/29 Long Term Auction
1,3210,PJM,Long Term 2026-29 RD 5 Credit Study,2026-03-03 00:00:00.000,2026-03-05 00:00:00.000,5,2026-06-01 00:00:00.000,CLOSED,2026-03-11 00:00:00.000,26/29 Long Term Auction Credit Study
2,3219,PJM,FEB 2026 Auction,2026-01-16 00:00:00.000,2026-01-21 00:00:00.000,1,2026-02-01 00:00:00.000,CLOSED,2026-01-28 00:00:00.000,FEB 2026 Auction
3,3220,PJM,MAR 2026 Auction,2026-02-11 00:00:00.000,2026-02-13 00:00:00.000,1,2026-03-01 00:00:00.000,CLOSED,2026-02-21 00:00:00.000,MAR 2026 Auction
4,3221,PJM,APR 2026 Auction,2026-03-12 00:00:00.000,2026-03-16 00:00:00.000,1,2026-04-01 00:00:00.000,OPEN,2026-03-23 00:00:00.000,APR 2026 Auction


In [ ]:
FTRAPeriods = pd.read_csv(f'data/FTRAuction/FTRAuctionPeriods_{today_str}.csv', skiprows=[1])

FTRAPeriods.head()

,AuctionKey,PeriodCode,StartDate,EndDate,PeakHours,OffpeakHours,H24Hours,PeriodTypeCode,ISOCode,DailyOffPeakHours,WkndOnPeakHours,ISOPeriodCode,PNodeBidType
0,3117,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
1,3118,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
2,3119,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
3,3120,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM
4,3121,PJM-YR-26-27,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000,4096.0,4664.0,8760.0,ANNUAL,PJM,2920.0,1744.0,YR3,NM


In [4]:
NodePrices = pd.read_csv(f'data/FTRAuction/FTRAuctionNodePrices_{today_str}.csv', skiprows=[1])

NodePrices.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,CongOnPeakMwh,CongOffPeakMwh,ISOCode,CongDailyOffPeakMwh,CongWkndOnPeakMwh,UploadDate
0,3117,PJM-YR-24-25,3,-1.875437,-2.660866,-1.195713,PJM,-0.847000,-1.769049,2023-06-12 16:46:30.857
1,3117,PJM-YR-24-25,4,4.054796,5.666407,2.660079,PJM,1.993459,3.756098,2023-06-12 16:46:30.857
2,3117,PJM-YR-24-25,5,4.495161,5.970529,3.218352,PJM,2.562705,4.296329,2023-06-12 16:46:30.857
3,3117,PJM-YR-24-25,6,10.011016,13.348408,7.122779,PJM,5.465387,9.847771,2023-06-12 16:46:30.857
4,3117,PJM-YR-24-25,7,-0.760172,1.043428,-2.321037,PJM,-3.586514,-0.240411,2023-06-12 16:46:30.857


In [5]:
NodePrices_mapped = (
    NodePrices[
        ["AuctionKey", "PeriodCode", "NodeKey", "Cong24HMwh"]
    ]
    .merge(
        FTRAuctions[
            ["AuctionKey", "AuctionName", "AuctionEndDate"]
        ],
        on="AuctionKey",
        how="inner"
    )
    .merge(
        FTRAPeriods[
            ["AuctionKey", "PeriodCode", "StartDate", "EndDate"]
        ],
        on=["AuctionKey", "PeriodCode"],
        how="inner"
    )
)

NodePrices_mapped.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
0,3205,PJM-YR-26-27,3,-0.191220,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
1,3205,PJM-YR-26-27,4,4.888166,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
2,3205,PJM-YR-26-27,5,8.538418,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
3,3205,PJM-YR-26-27,6,19.659943,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000
4,3205,PJM-YR-26-27,7,-3.970276,Long Term 2026-29 RD 5,2026-03-05 00:00:00.000,2026-06-01 00:00:00.000,2027-05-31 00:00:00.000


In [6]:
df = NodePrices_mapped.copy()

df["AuctionEndDate"] = pd.to_datetime(df["AuctionEndDate"])
df["StartDate"] = pd.to_datetime(df["StartDate"])
df["EndDate"] = pd.to_datetime(df["EndDate"])

# Extract auction month from names like "FEB 2026 Auction"
auction_parts = df["AuctionName"].str.extract(
    r"([A-Z]{3})\s+(\d{4})"
)

df["AuctionMonth"] = pd.to_datetime(
    auction_parts[0] + " " + auction_parts[1],
    format="%b %Y",
    errors="coerce"
)

auction_price = df[
    # monthly periods only
    (df["StartDate"].dt.year == 2026)
    & (df["EndDate"].dt.year == 2026)
    & (df["StartDate"].dt.month == df["EndDate"].dt.month)

    # # auction month = previous month of delivery month
    # & (
    #     df["AuctionMonth"].dt.to_period("M")
    #     == (df["StartDate"].dt.to_period("M"))
    # )
].copy()

auction_price = auction_price[
    [
        "AuctionKey",
        "PeriodCode",
        "NodeKey",
        "Cong24HMwh",
        "AuctionName",
        "AuctionEndDate",
        "StartDate",
        "EndDate",
    ]
]

auction_price.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
46599,3219,PJM-APR-26,3,0.123500,FEB 2026 Auction,2026-01-21,2026-04-01,2026-04-30
46600,3219,PJM-APR-26,4,7.838319,FEB 2026 Auction,2026-01-21,2026-04-01,2026-04-30
46601,3219,PJM-APR-26,5,7.946819,FEB 2026 Auction,2026-01-21,2026-04-01,2026-04-30
46602,3219,PJM-APR-26,6,18.232403,FEB 2026 Auction,2026-01-21,2026-04-01,2026-04-30
46603,3219,PJM-APR-26,7,-5.700306,FEB 2026 Auction,2026-01-21,2026-04-01,2026-04-30


In [7]:
last_auction_price =auction_price[(auction_price['AuctionName'] == "APR 2026 Auction") & (auction_price['StartDate'] == "2026-04-01")].copy()
last_auction_price.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
153639,3221,PJM-APR-26,3,-10.274652,APR 2026 Auction,2026-03-16,2026-04-01,2026-04-30
153640,3221,PJM-APR-26,4,0.997444,APR 2026 Auction,2026-03-16,2026-04-01,2026-04-30
153641,3221,PJM-APR-26,5,2.158125,APR 2026 Auction,2026-03-16,2026-04-01,2026-04-30
153642,3221,PJM-APR-26,6,14.554013,APR 2026 Auction,2026-03-16,2026-04-01,2026-04-30
153643,3221,PJM-APR-26,7,-14.686931,APR 2026 Auction,2026-03-16,2026-04-01,2026-04-30


In [8]:
auction_price[auction_price["StartDate"] == "2026-05-01"].sort_values("NodeKey").head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
92472,3219,PJM-MAY-26,3,0.671452,FEB 2026 Auction,2026-01-21,2026-05-01,2026-05-31
169177,3221,PJM-MAY-26,3,-13.486303,APR 2026 Auction,2026-03-16,2026-05-01,2026-05-31
184715,3222,PJM-MAY-26,3,-0.621129,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31
138347,3220,PJM-MAY-26,3,0.539099,MAR 2026 Auction,2026-02-13,2026-05-01,2026-05-31
92473,3219,PJM-MAY-26,4,9.298562,FEB 2026 Auction,2026-01-21,2026-05-01,2026-05-31


In [9]:
auction_price = auction_price[auction_price["AuctionName"] == "MAY 2026 Auction"].copy()

In [10]:
auction_price.head()

,AuctionKey,PeriodCode,NodeKey,Cong24HMwh,AuctionName,AuctionEndDate,StartDate,EndDate
184715,3222,PJM-MAY-26,3,-0.621129,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31
184716,3222,PJM-MAY-26,4,11.793360,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31
184717,3222,PJM-MAY-26,5,13.080550,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31
184718,3222,PJM-MAY-26,6,28.059704,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31
184719,3222,PJM-MAY-26,7,0.389866,MAY 2026 Auction,2026-04-20,2026-05-01,2026-05-31


In [11]:
auction_price['NodeKey'].nunique()

15538

### congestion spread

In [18]:
congestion_spread = pd.read_csv('data/nodeDALMPH/negative_unilateral_extreme_pairs_congestion_spread.csv')

In [19]:
congestion_spread["pair"].nunique()

830

In [20]:
congestion_spread.head()

,date,nodekey1,nodekey2,pair,distance,congestion_1,congestion_2,congestion_spread,nodename1,lat1,lon1,nodename2,lat2,lon2
0,2025-04-01 00:00:00,517,321988,517-321988,5.463062,-2.41,-2.49,0.08,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
1,2025-04-01 01:00:00,517,321988,517-321988,5.463062,-5.67,-5.95,0.28,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
2,2025-04-01 02:00:00,517,321988,517-321988,5.463062,-7.09,-7.44,0.35,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
3,2025-04-01 03:00:00,517,321988,517-321988,5.463062,-7.40,-7.77,0.37,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
4,2025-04-01 04:00:00,517,321988,517-321988,5.463062,-7.51,-7.88,0.37,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556


In [21]:
# fraction of zero spreads for each pair
zero_ratio = (
    congestion_spread
    .groupby("pair")["congestion_spread"]
    .apply(lambda x: (x == 0).mean())
)

# pairs to keep (<= 99% zeros)
valid_pairs = zero_ratio[zero_ratio <= 0.99].index

# filter dataset
congestion_spread = congestion_spread[
    congestion_spread["pair"].isin(valid_pairs)
].copy()

In [22]:
df = congestion_spread.copy()

# ----------------------------------
# Use history only before 2026-04-15
# ----------------------------------
hist_df = df.loc[
    df["date"] < "2026-04-15"
].copy()

def max_drawdown(x):
    x = pd.Series(x).fillna(0)

    cumulative = x.cumsum()
    running_max = cumulative.cummax()
    drawdown = cumulative - running_max

    return drawdown.min()


def cvar_1(x):
    x = pd.Series(x).dropna()

    if len(x) == 0:
        return np.nan

    q01 = x.quantile(0.01)

    # downside CVaR: average of worst 5% congestion_spread values
    return x[x <= q01].mean()


pair_risk = (
    hist_df
    .sort_values(["pair", "date"])
    .groupby("pair")["congestion_spread"]
    .agg(
        max_drawdown=max_drawdown,
        cvar_1=cvar_1
    )
    .reset_index()
)

# Worst 10% by max drawdown
dd_threshold = pair_risk["max_drawdown"].quantile(0.10)

worst_dd_pairs = pair_risk.loc[
    pair_risk["max_drawdown"] <= dd_threshold,
    "pair"
]

# Worst 10% by CVaR 1%
cvar_threshold = pair_risk["cvar_1"].quantile(0.10)

worst_cvar_pairs = pair_risk.loc[
    pair_risk["cvar_1"] <= cvar_threshold,
    "pair"
]

# Union of both high-risk groups
worst_pairs = pd.concat(
    [worst_dd_pairs, worst_cvar_pairs]
).drop_duplicates()

congestion_spread = df.loc[
    ~df["pair"].isin(worst_pairs)
].copy()

congestion_spread.head()

,date,nodekey1,nodekey2,pair,distance,congestion_1,congestion_2,congestion_spread,nodename1,lat1,lon1,nodename2,lat2,lon2
0,2025-04-01 00:00:00,517,321988,517-321988,5.463062,-2.41,-2.49,0.08,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
1,2025-04-01 01:00:00,517,321988,517-321988,5.463062,-5.67,-5.95,0.28,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
2,2025-04-01 02:00:00,517,321988,517-321988,5.463062,-7.09,-7.44,0.35,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
3,2025-04-01 03:00:00,517,321988,517-321988,5.463062,-7.40,-7.77,0.37,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556
4,2025-04-01 04:00:00,517,321988,517-321988,5.463062,-7.51,-7.88,0.37,125 NORM138 KV TR77 34,41.621187,-89.723244,980 WALN34.5 KV WALNTRWF,41.700096,-89.716556


In [23]:
# ---------------------------------------
# Split pair into nodekey1 / nodekey2
# ---------------------------------------
pairs = (
    congestion_spread[
        ["pair", "nodekey1", "nodekey2"]
    ]
    .drop_duplicates()
    .copy()
)

pairs["nodekey1"] = pairs["nodekey1"].astype(int)
pairs["nodekey2"] = pairs["nodekey2"].astype(int)

# ---------------------------------------
# Auction price for source node
# ---------------------------------------
auction_spread = pairs.merge(
    auction_price[
        ["AuctionName", "StartDate", "EndDate", "NodeKey", "Cong24HMwh"]
    ].rename(
        columns={
            "NodeKey": "nodekey1",
            "Cong24HMwh": "auction_cong_1"
        }
    ),
    on="nodekey1",
    how="inner"
)

# ---------------------------------------
# Auction price for sink node
# ---------------------------------------
auction_spread = auction_spread.merge(
    auction_price[
        ["AuctionName", "NodeKey", "Cong24HMwh"]
    ].rename(
        columns={
            "NodeKey": "nodekey2",
            "Cong24HMwh": "auction_cong_2"
        }
    ),
    on=["AuctionName", "nodekey2"],
    how="inner"
)

# ---------------------------------------
# Calculate auction spread
# ---------------------------------------
auction_spread["auction_spread"] = (
    auction_spread["auction_cong_1"]
    - auction_spread["auction_cong_2"]
)

auction_spread.head()

,pair,nodekey1,nodekey2,AuctionName,StartDate,EndDate,auction_cong_1,auction_cong_2,auction_spread
0,517-321988,517,321988,MAY 2026 Auction,2026-05-01,2026-05-31,-0.535121,-0.553925,0.018804
1,517-518,517,518,MAY 2026 Auction,2026-05-01,2026-05-31,-0.535121,-0.391815,-0.143306
2,517-324363,517,324363,MAY 2026 Auction,2026-05-01,2026-05-31,-0.535121,-1.069718,0.534597
3,969-4386,969,4386,MAY 2026 Auction,2026-05-01,2026-05-31,1.907204,1.907204,0.000000
4,969-4398,969,4398,MAY 2026 Auction,2026-05-01,2026-05-31,1.907204,2.156680,-0.249476


In [24]:
auction_spread['pair'].nunique()

722

### reference price 1

In [25]:
df = congestion_spread.copy()

df["date"] = pd.to_datetime(df["date"])

# ----------------------------------
# Historical months used for bootstrap
# ----------------------------------
bootstrap_df = df[
    (
        ((df["date"].dt.year == 2025) &
        (df["date"].dt.month.isin([4, 5, 6, 7])))
         # (df["date"].dt.month.isin([4, 5, 6])))
    )
    |
    (
        ((df["date"].dt.year == 2026) &
        (df["date"].dt.month == 4) &
        (df["date"].dt.day <= 15))
    )
].copy()

# ----------------------------------
# Double-weight May-2025
# ----------------------------------
may_2025 = bootstrap_df[
    (bootstrap_df["date"].dt.year == 2025)
    & (bootstrap_df["date"].dt.month == 5)
]

bootstrap_df = pd.concat(
    [bootstrap_df, may_2025],
    ignore_index=True
)

# apr_2026 = bootstrap_df[
#     (bootstrap_df["date"].dt.year == 2026)
#     & (bootstrap_df["date"].dt.month == 4)
# ]

# bootstrap_df = pd.concat(
#     [bootstrap_df, apr_2026],
#     ignore_index=True
# )

# ----------------------------------
# Forecast settings
# ----------------------------------
hours_in_may_2026 = 31 * 24
n_sims = 1000

results = []

# ----------------------------------
# Pair-by-pair bootstrap
# ----------------------------------
for pair_id, pair_df in bootstrap_df.groupby("pair"):

    spreads = pair_df["congestion_spread"].dropna().values
    
    # 10% lower tails
    q10 = np.quantile(spreads, 0.10)
    # q90 = np.quantile(spreads, 0.90)

    tail_data = spreads[(spreads <= q10)]

    # # duplicate tails once
    # spreads = np.concatenate([
    #     spreads,
    #     tail_data
    #     # np.repeat(tail_data, 3)
    # ])
    
    if len(spreads) < 100:
        continue

    # shape = (744 hours, 1000 simulations)
    simulations = np.random.choice(
        spreads,
        size=(hours_in_may_2026, n_sims),
        replace=True
    )

    # monthly average spread for each simulation
    sim_monthly_avg = simulations.mean(axis=0)

    reference_price = sim_monthly_avg.mean()

    results.append({
        "pair": pair_id,
        "reference_price": reference_price,
        "sim_std": sim_monthly_avg.std(),
        "sim_p05": np.quantile(sim_monthly_avg, 0.05),
        "sim_p50": np.quantile(sim_monthly_avg, 0.50),
        "sim_p95": np.quantile(sim_monthly_avg, 0.95),
        "n_hist_obs": len(spreads)
    })

reference_prices = pd.DataFrame(results)

reference_prices.head()

,pair,reference_price,sim_std,sim_p05,sim_p50,sim_p95,n_hist_obs
0,1256-471,-6.555145,0.993015,-8.237954,-6.535874,-5.001558,4032
1,1256-969,-5.344334,0.802882,-6.671560,-5.302372,-4.076468,4032
2,1291-969,-6.916278,1.085135,-8.778356,-6.870168,-5.203961,4032
3,13106-3270,-2.220137,0.492234,-3.061342,-2.196257,-1.435905,4032
4,13106-7075,-0.490395,0.100400,-0.663448,-0.489879,-0.328958,4032


In [41]:
execution_spread = (
    auction_spread
    .merge(
        reference_prices[["pair", "reference_price"]],
        on="pair",
        how="inner"
    )
    .assign(
        reference_price=lambda x: x["reference_price"].round(1)
    )
    .query(
        "(reference_price > auction_spread)"
    )
    .copy()
)

execution_spread.head()

,pair,nodekey1,nodekey2,AuctionName,StartDate,EndDate,auction_cong_1,auction_cong_2,auction_spread,reference_price
5,1566-6110,1566,6110,MAY 2026 Auction,2026-05-01,2026-05-31,8.357151,13.462271,-5.105120,-0.8
6,1566-5354,1566,5354,MAY 2026 Auction,2026-05-01,2026-05-31,8.357151,13.490632,-5.133481,-0.8
9,1629-1769,1629,1769,MAY 2026 Auction,2026-05-01,2026-05-31,19.426491,50.078484,-30.651993,-20.9
10,1768-1770,1768,1770,MAY 2026 Auction,2026-05-01,2026-05-31,45.449760,83.006454,-37.556694,-5.7
11,1768-323914,1768,323914,MAY 2026 Auction,2026-05-01,2026-05-31,45.449760,87.601219,-42.151459,-6.8


In [42]:
execution_spread['pair'].nunique()

311

### reference price 2

In [ ]:
optimal_rows = []

congestion_spread["date"] = pd.to_datetime(congestion_spread["date"])

# Select May 2025
may_2025 = congestion_spread[
    congestion_spread["date"].between(
        pd.Timestamp("2025-05-01"),
        pd.Timestamp("2025-05-31 23:59:59")
    )
].copy()

# Append a duplicate
filtered_congestion_spread = pd.concat(
    [congestion_spread, may_2025],
    ignore_index=True
)

filtered_congestion_spread = filtered_congestion_spread[
    (
        filtered_congestion_spread["date"].between(
            pd.Timestamp("2025-04-01"),
            pd.Timestamp("2025-07-31 23:59:59")
        )
    )
    |
    (
        filtered_congestion_spread["date"].between(
            pd.Timestamp("2026-04-01"),
            pd.Timestamp("2026-04-15 23:59:59")
        )
    )
].copy()

for pair_id in filtered_congestion_spread["pair"].astype(str).unique():

    pair_df = filtered_congestion_spread[
        filtered_congestion_spread["pair"].astype(str) == pair_id
    ].copy()

    cs = pair_df["congestion_spread"].dropna().values

    # skip pairs without enough data
    if len(cs) < 10:
        continue

    q05 = np.quantile(cs, 0.02)
    q95 = np.quantile(cs, 0.98)

    p_values = np.linspace(q05, q95, 500)

    profit_means = []
    loss_means = []

    for p in p_values:

        profit = cs[cs > p] - p
        loss = p - cs[cs < p]

        if len(profit) == 0:
            profit_mean = np.nan
            loss_mean = np.nan
        else:
            profit_mean = profit.mean()

            if len(loss) > 0:
                loss_mean = loss.mean() * len(loss) / len(profit)
            else:
                loss_mean = 0

        profit_means.append(profit_mean)
        loss_means.append(loss_mean)

    plot_df = pd.DataFrame({
        "reference_price": p_values,
        "profit_mean": profit_means,
        "loss_mean": loss_means
    })

    if plot_df[["profit_mean", "loss_mean"]].dropna().empty:
        continue

    idx = np.nanargmin(
        np.abs(plot_df["profit_mean"] - plot_df["loss_mean"])
    )

    optimal_p = plot_df.iloc[idx]["reference_price"]

    optimal_rows.append({
        "pair": pair_id,
        "optimal_reference_price": optimal_p,
        "q05": q05,
        "q95": q95,
        "n_obs": len(cs)
    })

optimal_p_by_pair = pd.DataFrame(optimal_rows)

optimal_p_by_pair = optimal_p_by_pair.sort_values(
    "optimal_reference_price",
    ascending=False
).reset_index(drop=True)

optimal_p_by_pair.head()

In [ ]:
reference_price_2 = (
    optimal_p_by_pair[
        ["pair", "optimal_reference_price"]
    ]
    .rename(
        columns={
            "optimal_reference_price": "reference_price_2"
        }
    )
)

reference_price_2.head()

In [ ]:
execution_spread = (
    auction_spread
    .merge(
        reference_price_2[["pair", "reference_price_2"]],
        on="pair",
        how="inner"
    )
    .query("reference_price_2 > auction_spread")
    .copy()
)

execution_spread.head()

In [ ]:
execution_spread['pair'].nunique()

### reference price 3

In [43]:

results = []

congestion_spread["date"] = pd.to_datetime(congestion_spread["date"])

# Select May 2025
may_2025 = congestion_spread[
    congestion_spread["date"].between(
        pd.Timestamp("2025-05-01"),
        pd.Timestamp("2025-05-31 23:59:59")
    )
].copy()

# Append a duplicate
filtered_congestion_spread = pd.concat(
    [congestion_spread, may_2025],
    ignore_index=True
)

filtered_congestion_spread = filtered_congestion_spread[
    (
        filtered_congestion_spread["date"].between(
            pd.Timestamp("2025-04-01"),
            pd.Timestamp("2025-06-30 23:59:59")
        )
    )
    |
    (
        filtered_congestion_spread["date"].between(
            pd.Timestamp("2026-04-01"),
            pd.Timestamp("2026-04-15 23:59:59")
        )
    )
].copy()

def power_tail_utility(x, delta, alpha):
    x = np.asarray(x)

    u = np.empty_like(x, dtype=float)

    mask = x < delta

    u[mask] = x[mask]

    u[~mask] = (
        delta
        + (x[~mask] - delta) ** alpha
    )

    return u

# -----------------------------
# Gain utility
# -----------------------------
def utility_gain(x, delta_gain=10, alpha_gain=0.5):

    x = np.asarray(x)

    base = power_tail_utility(
        x,
        delta=delta_gain,
        alpha=alpha_gain
    )

    return np.maximum(base, 0) 


# -----------------------------
# Loss utility
# -----------------------------
def utility_loss(x, delta_loss=10, alpha_loss=1.2):

    x = np.asarray(x)

    base = power_tail_utility(
        x,
        delta=delta_loss,
        alpha=alpha_loss
    )

    return np.maximum(base, 0)


for pair_id, pair_df in filtered_congestion_spread.groupby("pair"):

    pair_df = filtered_congestion_spread[
        filtered_congestion_spread["pair"].astype(str) == pair_id
    ].copy()

    cs = pair_df["congestion_spread"].dropna().values

    # 10% lower and upper tails
    q10 = np.quantile(cs, 0.10)

    tail_data = cs[(cs <= q10)]

    # duplicate tails once
    cs = np.concatenate([
        cs,
        tail_data
    ])
    

    if len(cs) < 10:
        continue

    q05 = np.quantile(cs, 0.05)
    q95 = np.quantile(cs, 0.95)

    if q05 == q95:
        continue

    p_values = np.linspace(q05, q95, 500)

    profit_utilities = []
    loss_utilities = []

    for p in p_values:

        profit = cs[cs > p] - p
        loss = p - cs[cs < p]

        if len(profit) == 0:
            profit_utility = np.nan
            loss_utility = np.nan
        else:
            profit_utility = (
                utility_gain(profit).mean()
            )

            if len(loss) > 0:
                loss_utility = (
                    utility_loss(loss).mean() * len(loss) / len(profit)
                )
            else:
                loss_utility = 0

        profit_utilities.append(profit_utility)
        loss_utilities.append(loss_utility)

    pair_result_df = pd.DataFrame({
        "reference_price": p_values,
        "profit_utility": profit_utilities,
        "loss_utility": loss_utilities
    })

    if pair_result_df[["profit_utility", "loss_utility"]].dropna().empty:
        continue

    idx = np.nanargmin(
        np.abs(
            pair_result_df["profit_utility"]
            - pair_result_df["loss_utility"]
        )
    )

    optimal_p = pair_result_df.iloc[idx]["reference_price"]

    results.append({
        "pair": pair_id,
        "optimal_reference_price": optimal_p,
        "profit_utility": pair_result_df.iloc[idx]["profit_utility"],
        "loss_utility": pair_result_df.iloc[idx]["loss_utility"],
        "utility_gap": abs(
            pair_result_df.iloc[idx]["profit_utility"]
            - pair_result_df.iloc[idx]["loss_utility"]
        ),
        "n_obs": len(cs),
        "q05": q05,
        "q95": q95
    })


utility_optimal_prices_all_pairs = pd.DataFrame(results)

utility_optimal_prices_all_pairs = (
    utility_optimal_prices_all_pairs
    .sort_values("optimal_reference_price")
    .reset_index(drop=True)
)

utility_optimal_prices_all_pairs.head()

,pair,optimal_reference_price,profit_utility,loss_utility,utility_gap,n_obs,q05,q95
0,1629-1769,-169.670002,21.770690,23.553197,1.782508,3617,-169.670002,0.80
1,2993-1769,-165.039993,21.571513,22.780983,1.209471,3617,-165.039993,1.20
2,2991-1769,-165.039993,21.571513,22.780983,1.209471,3617,-165.039993,1.20
3,1821-6240,-163.839993,21.383350,22.560787,1.177437,3617,-163.839993,2.11
4,2280-1566,-129.212022,19.871918,19.902203,0.030286,3617,-211.149996,0.70


In [44]:
reference_price_3 = (
    utility_optimal_prices_all_pairs[
        ["pair", "optimal_reference_price"]
    ]
    .rename(
        columns={
            "optimal_reference_price": "reference_price_3"
            
        }
    )
)

reference_price_3.head()

,pair,reference_price_3
0,1629-1769,-169.670002
1,2993-1769,-165.039993
2,2991-1769,-165.039993
3,1821-6240,-163.839993
4,2280-1566,-129.212022


In [67]:
execution_spread = (
    auction_spread
    .merge(
        reference_prices[["pair", "reference_price"]],
        on="pair",
        how="inner"
    )
    .assign(
        reference_price=lambda x: x["reference_price"].round(1)
    )
    .query("(reference_price > auction_spread)")
    .merge(
        reference_price_3[["pair", "reference_price_3"]],
        on="pair",
        how="inner"
    )
    .assign(
        reference_price_3=lambda x: x["reference_price_3"].round(1)
    )
    .query("(reference_price_3 > auction_spread)") 
    .copy()
)

execution_spread.head()

,pair,nodekey1,nodekey2,AuctionName,StartDate,EndDate,auction_cong_1,auction_cong_2,auction_spread,reference_price,reference_price_3
0,1566-6110,1566,6110,MAY 2026 Auction,2026-05-01,2026-05-31,8.357151,13.462271,-5.105120,-0.8,-1.8
1,1566-5354,1566,5354,MAY 2026 Auction,2026-05-01,2026-05-31,8.357151,13.490632,-5.133481,-0.8,-1.9
3,1768-323914,1768,323914,MAY 2026 Auction,2026-05-01,2026-05-31,45.449760,87.601219,-42.151459,-6.8,-2.0
4,1770-323914,1770,323914,MAY 2026 Auction,2026-05-01,2026-05-31,83.006454,87.601219,-4.594765,-1.1,-0.4
12,2253-2936,2253,2936,MAY 2026 Auction,2026-05-01,2026-05-31,-0.645914,-0.520739,-0.125175,0.0,-0.1


In [ ]:
# df = execution_spread.sample(frac=1, random_state=42).copy()

# node_counts = {}
# keep_idx = []

# for idx, row in df.iterrows():
#     n1 = row["nodekey1"]
#     n2 = row["nodekey2"]

#     if node_counts.get(n1, 0) < 3 and node_counts.get(n2, 0) < 3:
#         keep_idx.append(idx)
#         node_counts[n1] = node_counts.get(n1, 0) + 1
#         node_counts[n2] = node_counts.get(n2, 0) + 1

# execution_spread = (
#     execution_spread
#     .loc[keep_idx]
#     .sort_index()
#     .copy()
# )

In [72]:
execution_spread['pair'].nunique()

75

### execution pair

In [74]:
# execution_spread = auction_spread.copy()

exec_pairs = execution_spread["pair"].astype(str).unique()

cs_may = congestion_spread.copy()

cs_may["date"] = pd.to_datetime(cs_may["date"])
cs_may["pair"] = cs_may["pair"].astype(str)

cs_may = cs_may.loc[
    (cs_may["pair"].isin(exec_pairs))
    & (cs_may["date"] >= "2026-05-01")
    & (cs_may["date"] < "2026-06-01")
].copy()

auction_info = execution_spread[
    ["pair", "auction_spread", "AuctionName"] # reference price
].copy()

auction_info["pair"] = auction_info["pair"].astype(str)

pnl_df = cs_may.merge(
    auction_info,
    on="pair",
    how="inner"
)

# Calculate hourly PnL
pnl_df["pnl"] = (
    pnl_df["congestion_spread"] - pnl_df["auction_spread"]
)

pnl_df = pnl_df.sort_values(["pair", "date"])

pnl_df["cum_pnl"] = (
    pnl_df
    .groupby("pair")["pnl"]
    .cumsum()
)

# Optional display label
pnl_df["pair_label"] = (
    pnl_df["pair"].astype(str)
    + " | "
    + pnl_df["nodename1"].astype(str)
    + " -> "
    + pnl_df["nodename2"].astype(str)
)

In [75]:
# from plotly.subplots import make_subplots
# import plotly.graph_objects as go

# fig = make_subplots(
#     rows=2,
#     cols=1,
#     shared_xaxes=True,
#     vertical_spacing=0.08,
#     subplot_titles=(
#         "Hourly PnL",
#         "Cumulative PnL"
#     )
# )

# pairs = pnl_df["pair"].drop_duplicates().tolist()

# for i, pair_id in enumerate(pairs):

#     df_pair = (
#         pnl_df[pnl_df["pair"] == pair_id]
#         .sort_values("date")
#         .copy()
#     )

#     label = df_pair["pair_label"].iloc[0]

#     # Hourly PnL
#     fig.add_trace(
#         go.Scatter(
#             x=df_pair["date"],
#             y=df_pair["pnl"],
#             mode="lines",
#             name="Hourly PnL",
#             visible=(i == 0),
#             hovertemplate=(
#                 "Date: %{x}<br>"
#                 "Hourly PnL: %{y:.4f}<br>"
#                 "<extra></extra>"
#             )
#         ),
#         row=1,
#         col=1
#     )

#     # Cumulative PnL
#     fig.add_trace(
#         go.Scatter(
#             x=df_pair["date"],
#             y=df_pair["cum_pnl"],
#             mode="lines",
#             name="Cumulative PnL",
#             visible=(i == 0),
#             hovertemplate=(
#                 "Date: %{x}<br>"
#                 "Cumulative PnL: %{y:.4f}<br>"
#                 "<extra></extra>"
#             )
#         ),
#         row=2,
#         col=1
#     )

# # -----------------------------
# # Dropdown
# # -----------------------------
# buttons = []

# for i, pair_id in enumerate(pairs):

#     visible = [False] * (2 * len(pairs))
#     visible[2 * i] = True
#     visible[2 * i + 1] = True

#     df_pair = pnl_df[pnl_df["pair"] == pair_id]
#     label = df_pair["pair_label"].iloc[0]

#     buttons.append(
#         dict(
#             label=label,
#             method="update",
#             args=[
#                 {"visible": visible},
#                 {
#                     "title": (
#                         f"May 2026 PnL: {label}<br>"
#                         "Auction Spread - Congestion Spread"
#                     )
#                 }
#             ]
#         )
#     )

# fig.update_layout(
#     title="May 2026 PnL by Pair",
#     template="plotly_white",
#     hovermode="x unified",
#     height=800,
#     updatemenus=[
#         dict(
#             buttons=buttons,
#             direction="down",
#             showactive=True,
#             x=0.01,
#             y=1.20,
#             xanchor="left",
#             yanchor="top"
#         )
#     ],
#     legend=dict(
#         orientation="h",
#         y=1.05
#     )
# )

# fig.update_xaxes(title_text="Date", row=2, col=1)
# fig.update_yaxes(title_text="Hourly PnL", row=1, col=1)
# fig.update_yaxes(title_text="Cumulative PnL", row=2, col=1)

# fig.show()

In [76]:
from dash import Dash, dash_table, dcc, html, Input, Output

# -----------------------------------
# 1. Prepare summary table
# -----------------------------------
pnl_df["date"] = pd.to_datetime(pnl_df["date"])
pnl_df["pair"] = pnl_df["pair"].astype(str)

pair_summary = (
    pnl_df.groupby(["pair", "pair_label"])
    .agg(
        auction_spread=("auction_spread", "first"),
        total_pnl=("pnl", "sum"),
        avg_pnl=("pnl", "mean"),
        hit_rate=("pnl", lambda x: (x > 0).mean()),
        max_drawdown=("cum_pnl", lambda x: (x - x.cummax()).min()),
        last_cum_pnl=("cum_pnl", "last")
    )
    .reset_index()
)

pair_summary["hit_rate"] = pair_summary["hit_rate"] * 100

pair_summary_display = pair_summary.copy()

for col in [
    "auction_spread",
    "total_pnl",
    "avg_pnl",
    "max_drawdown",
    "last_cum_pnl",
    "hit_rate"
]:
    pair_summary_display[col] = pair_summary_display[col].round(2)

# -----------------------------------
# 2. Function to create chart
# -----------------------------------
def make_pair_chart(pair_id):

    df_pair = (
        pnl_df[pnl_df["pair"] == pair_id]
        .sort_values("date")
        .copy()
    )

    label = df_pair["pair_label"].iloc[0]

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df_pair["date"],
            y=df_pair["pnl"],
            mode="lines",
            name="Hourly PnL",
            hovertemplate=(
                "Date: %{x}<br>"
                "Hourly PnL: %{y:.4f}<br>"
                "<extra></extra>"
            )
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df_pair["date"],
            y=df_pair["cum_pnl"],
            mode="lines",
            name="Cumulative PnL",
            hovertemplate=(
                "Date: %{x}<br>"
                "Cumulative PnL: %{y:.4f}<br>"
                "<extra></extra>"
            )
        )
    )

    # auction_spread = df_pair["auction_spread"].iloc[0]

    # fig.add_trace(
    #     go.Scatter(
    #         x=df_pair["date"],
    #         y=[auction_spread] * len(df_pair),
    #         mode="lines",
    #         name=f"Auction Spread: {auction_spread:.2f}",
    #         line=dict(dash="dash"),
    #         hovertemplate=(
    #             "Date: %{x}<br>"
    #             "Auction Spread: %{y:.4f}<br>"
    #             "<extra></extra>"
    #         )
    #     )
    # )

    fig.update_layout(
        title=f"May 2026 PnL: {label}",
        xaxis_title="Date",
        yaxis_title="PnL",
        hovermode="x unified",
        template="plotly_white",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=50, r=30, t=70, b=40)
    )

    return fig


# -----------------------------------
# 3. Dash app
# -----------------------------------
app = Dash(__name__)

default_pair = pair_summary_display.iloc[0]["pair"]

app.layout = html.Div(
    style={
        "display": "grid",
        "gridTemplateColumns": "42% 58%",
        "gap": "18px",
        "height": "90vh",
        "padding": "16px"
    },
    children=[

        html.Div(
            children=[
                html.H3("Pair Ranking"),

                dash_table.DataTable(
                    id="pair-table",
                    data=pair_summary_display.to_dict("records"),
                    columns=[
                        {"name": "Pair", "id": "pair"},
                        {"name": "Pair Label", "id": "pair_label"},
                        {"name": "Auction Spread", "id": "auction_spread", "type": "numeric"},
                        {"name": "Total PnL", "id": "total_pnl", "type": "numeric"},
                        {"name": "Avg PnL", "id": "avg_pnl", "type": "numeric"},
                        {"name": "Hit Rate %", "id": "hit_rate", "type": "numeric"},
                        {"name": "Max Drawdown", "id": "max_drawdown", "type": "numeric"},
                        # {"name": "Last Cum PnL", "id": "last_cum_pnl", "type": "numeric"},
                    ],
                    page_size=20,
                    sort_action="native",
                    filter_action="native",
                    row_selectable="single",
                    selected_rows=[0],
                    style_table={
                        "height": "80vh",
                        "overflowY": "auto",
                        "overflowX": "auto"
                    },
                    style_cell={
                        "textAlign": "left",
                        "padding": "8px",
                        "fontFamily": "Arial",
                        "fontSize": "13px",
                        "minWidth": "90px",
                        "maxWidth": "220px",
                        "whiteSpace": "normal"
                    },
                    style_header={
                        "fontWeight": "bold",
                        "backgroundColor": "#f4f4f4"
                    },
                    style_data_conditional=[
                        {
                            "if": {"state": "selected"},
                            "backgroundColor": "#dbeafe",
                            "border": "1px solid #2563eb"
                        },
                        {
                            "if": {
                                "filter_query": "{total_pnl} > 0",
                                "column_id": "total_pnl"
                            },
                            "color": "green",
                            "fontWeight": "bold"
                        },
                        {
                            "if": {
                                "filter_query": "{total_pnl} < 0",
                                "column_id": "total_pnl"
                            },
                            "color": "red",
                            "fontWeight": "bold"
                        }
                    ]
                )
            ]
        ),

        html.Div(
            children=[
                dcc.Graph(
                    id="pnl-chart",
                    figure=make_pair_chart(default_pair),
                    style={"height": "86vh"}
                )
            ]
        )
    ]
)


# -----------------------------------
# 4. Callback: click table row -> update chart
# -----------------------------------
@app.callback(
    Output("pnl-chart", "figure"),
    Input("pair-table", "derived_virtual_data"),
    Input("pair-table", "derived_virtual_selected_rows")
)
def update_chart(rows, selected_rows):

    if rows is None:
        rows = pair_summary_display.to_dict("records")

    if selected_rows is None or len(selected_rows) == 0:
        pair_id = rows[0]["pair"]
    else:
        pair_id = rows[selected_rows[0]]["pair"]

    return make_pair_chart(pair_id)


# -----------------------------------
# 5. Run
# -----------------------------------
app.run(debug=True)

In [77]:
# Final cumulative pnl for each pair
final_pnl = (
    pnl_df
    .groupby("pair", as_index=False)
    .agg(
        final_cum_pnl=("cum_pnl", "last")
    )
)

# Count pairs ending positive
positive_count = (
    final_pnl["final_cum_pnl"] > 0
).sum()

total_pairs = len(final_pnl)

positive_rate = positive_count / total_pairs

print(f"Positive pairs: {positive_count}")
print(f"Total pairs: {total_pairs}")
print(f"Positive rate: {positive_rate:.2%}")

Positive pairs: 53
Total pairs: 75
Positive rate: 70.67%


In [78]:
# Top 5 PnL pairs
top_5_pnl = (
    final_pnl
    .sort_values("final_cum_pnl", ascending=False)
    .head(5)
)

# Bottom 5 PnL pairs
bottom_5_pnl = (
    final_pnl
    .sort_values("final_cum_pnl", ascending=True)
    .head(5)
)

print("Top 5 Pairs")
print(top_5_pnl)

print("\nBottom 5 Pairs")
print(bottom_5_pnl)

Top 5 Pairs
           pair  final_cum_pnl
69    6240-2527    4935.930356
74    8045-8452    3977.429905
56  324318-3059    3467.370089
2     1566-5354    2859.259846
3     1566-6110    2838.159262

Bottom 5 Pairs
            pair  final_cum_pnl
5    1770-323914  -13938.454459
42   31230-30993   -1744.470511
17   27834-27907   -1622.490966
55  323852-30993   -1485.009892
32   30970-27834   -1443.409485


In [79]:
candidate_pair = bottom_5_pnl.iloc[0]["pair"]

pair_df = congestion_spread[
    congestion_spread["pair"] == candidate_pair
].copy()

pair_df["date"] = pd.to_datetime(pair_df["date"])
pair_df = pair_df.sort_values("date")

fig = px.line(
    pair_df,
    x="date",
    y="congestion_spread",
    title="Pair Congestion Spread",
    labels={
        "date": "Date",
        "congestion_spread": "Congestion Spread"
    }
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [80]:
execution_spread[execution_spread["pair"] == candidate_pair].head()

,pair,nodekey1,nodekey2,AuctionName,StartDate,EndDate,auction_cong_1,auction_cong_2,auction_spread,reference_price,reference_price_3
4,1770-323914,1770,323914,MAY 2026 Auction,2026-05-01,2026-05-31,83.006454,87.601219,-4.594765,-1.1,-0.4


### portfolio

In [81]:
# Portfolio hourly PnL
portfolio_pnl = (
    pnl_df
    .groupby("date")["pnl"]
    .sum()
    .reset_index()
)

daily_pnl = (
    portfolio_pnl
    .set_index("date")["pnl"]
    .resample("D")
    .sum()
)

daily_hit_rate = (daily_pnl > 0).mean()

portfolio_pnl["cum_pnl"] = portfolio_pnl["pnl"].cumsum()

portfolio_pnl['drawdown'] = portfolio_pnl['cum_pnl'] - portfolio_pnl['cum_pnl'].cummax()

max_drawdown = portfolio_pnl['drawdown'].min()

def strategy_score(pnl_series):

    pnl = pd.Series(pnl_series).fillna(0)

    # cumulative pnl
    cum_pnl = pnl.cumsum()

    # drawdown
    running_max = cum_pnl.cummax()
    drawdown = cum_pnl - running_max

    max_drawdown = abs(drawdown.min())

    # recovery times
    underwater = cum_pnl < running_max

    recovery_times = []
    start = None

    for i, is_underwater in enumerate(underwater):

        if is_underwater and start is None:
            start = i

        elif not is_underwater and start is not None:
            recovery_times.append(i - start)
            start = None

    max_recovery_time = (
        max(recovery_times)
        if len(recovery_times) > 0
        else 0
    )

    avg_recovery_time = (
        np.mean(recovery_times)
        if len(recovery_times) > 0
        else 0
    )

    total_pnl = pnl.sum()

    hit_rate = (pnl > 0).mean()

    sharpe = (
        pnl.mean() / pnl.std()
        if pnl.std() > 0
        else 0
    )

    calmar = (
        total_pnl / max_drawdown
        if max_drawdown > 0
        else np.nan
    )

    score = (
        calmar / np.log1p(max_recovery_time)
        if max_recovery_time > 0
        else calmar
    )

    return {
        "total_pnl": total_pnl,
        "max_drawdown": max_drawdown,
        "hit_rate": hit_rate,
        "sharpe": sharpe,
        "calmar": calmar,
        "max_recovery_time": max_recovery_time,
        "avg_recovery_time": avg_recovery_time,
        "score": score
    }

stats = strategy_score(portfolio_pnl["pnl"])

print(f"""
Performance Summary
-------------------
Total PnL          : {stats['total_pnl']:,.2f}
Max Drawdown       : {stats['max_drawdown']:,.2f}
Hit Rate           : {stats['hit_rate']:.2%}
Daily Hit Rate     : {daily_hit_rate:.2%}
Sharpe Ratio       : {stats['sharpe']:,.2f}
Calmar Ratio       : {stats['calmar']:.3f}
Max Recovery Time  : {stats['max_recovery_time']} hours
Avg Recovery Time  : {stats['avg_recovery_time']:.1f} hours
Score              : {stats['score']:.4f}
""")


Performance Summary
-------------------
Total PnL          : 31,581.03
Max Drawdown       : 41,895.80
Hit Rate           : 82.53%
Daily Hit Rate     : 80.65%
Sharpe Ratio       : 0.14
Calmar Ratio       : 0.754
Max Recovery Time  : 42 hours
Avg Recovery Time  : 11.9 hours
Score              : 0.2004



In [82]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[
        "Hourly Portfolio PnL",
        "Cumulative Portfolio PnL",
        "Drawdown"
    ]
)

# Hourly PnL
fig.add_trace(
    go.Scatter(
        x=portfolio_pnl["date"],
        y=portfolio_pnl["pnl"],
        mode="lines",
        name="Hourly PnL"
    ),
    row=1,
    col=1
)

# Cumulative PnL
fig.add_trace(
    go.Scatter(
        x=portfolio_pnl["date"],
        y=portfolio_pnl["cum_pnl"],
        mode="lines",
        name="Cumulative PnL"
    ),
    row=2,
    col=1
)

# Drawdown
fig.add_trace(
    go.Scatter(
        x=portfolio_pnl["date"],
        y=portfolio_pnl["drawdown"],
        mode="lines",
        fill="tozeroy",
        name="Drawdown"
    ),
    row=3,
    col=1
)

final_pnl = portfolio_pnl["cum_pnl"].iloc[-1]

fig.add_trace(
    go.Scatter(
        x=[portfolio_pnl["date"].iloc[-1]],
        y=[final_pnl],
        mode="markers+text",
        text=[f"{final_pnl:,.0f}"],
        textposition="bottom right",
        marker=dict(
            size=10,
            symbol="diamond",
            color="red"
        ),
        name="Final PnL"
    ),
    row=2,
    col=1
)

# Mark max drawdown point
max_dd_idx = portfolio_pnl["drawdown"].idxmin()

fig.add_trace(
    go.Scatter(
        x=[portfolio_pnl.loc[max_dd_idx, "date"]],
        y=[portfolio_pnl.loc[max_dd_idx, "drawdown"]],
        mode="markers+text",
        text=[f"{max_drawdown:,.0f}"],
        textposition="bottom right",
        marker=dict(
            size=10,
            symbol="x"
        ),
        name="Max Drawdown"
    ),
    row=3,
    col=1
)

summary_text = (
    f"<b>Performance Summary</b><br>"
    f"Positive Pairs: {positive_count}<br>"
    f"Positive Rate: {positive_rate:.2%}<br>"
    f"Total PnL: {stats['total_pnl']:,.2f}<br>"
    f"Max Drawdown: {stats['max_drawdown']:,.2f}<br>"
    f"Calmar Ratio: {stats['calmar']:.3f}<br>"
    f"Hit Rate: {stats['hit_rate']:.2%}<br>"
    f"Daily Hit Rate: {daily_hit_rate:.2%}<br>"
    f"Sharpe Ratio: {stats['sharpe']:,.2f}<br>"
    f"Max Recovery Time: {stats['max_recovery_time']} hours<br>"
    f"Avg Recovery Time: {stats['avg_recovery_time']:.1f} hours<br>"
)

fig.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0,
    y=-0.30,
    showarrow=False,
    align="left",
    bordercolor="black",
    borderwidth=1,
    bgcolor="white",
    font=dict(size=12)
)


fig.update_layout(
    title="Portfolio Performance",
    hovermode="x unified",
    height=1050,
    showlegend=True,
    margin=dict(b=220)
)

fig.update_yaxes(title_text="Hourly PnL", row=1, col=1)
fig.update_yaxes(title_text="Cum PnL", row=2, col=1)
fig.update_yaxes(title_text="Drawdown", row=3, col=1)

fig.show()

In [83]:
# fig.write_html("graph/pair_congestion_spread/neutral_portfolio_performance.html")

# end